# YouTube Transcript to Structured Notes

## 1. Project Overview

**Task:** Take a raw YouTube video transcript and transform it into **structured notes, summary, key takeaways, action items, and quiz questions** — all using prompt engineering with a local LLM.

**Approach:**
1. Fetch the transcript via the YouTube Transcript API
2. Clean and preprocess the raw text
3. Use carefully designed prompts to extract different output formats
4. Compare prompt strategies (zero-shot vs few-shot, flat vs structured)

**Stack:**
- `youtube-transcript-api` — fetch transcripts without needing a Google API key
- `LangChain` + `ChatOllama` — prompt engineering and LLM interaction
- `Ollama` + `qwen3.5:9b` — local LLM (no cloud API needed)

> **No API keys required.** Transcripts are fetched from YouTube's public subtitle system. The LLM runs locally via Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Fetch and parse YouTube transcripts programmatically |
| 2 | Preprocess noisy transcript text (timestamps, filler words, formatting) |
| 3 | Design **task-specific prompts** that produce structured output |
| 4 | Understand the difference between summarization, extraction, and generation tasks |
| 5 | Use **output schemas** in prompts to control LLM response format |
| 6 | Handle long transcripts that exceed context windows via chunked summarization |
| 7 | Compare zero-shot vs few-shot prompting for structured extraction |
| 8 | Build a reusable note-generation pipeline |

## 3. Problem Statement

YouTube is one of the largest sources of educational content. However:
- Watching a 30-minute lecture takes 30 minutes
- Auto-generated subtitles are noisy (no punctuation, no paragraphs, filler words)
- There's no built-in way to get structured study notes
- Students often watch passively without creating actionable takeaways

**Goal:** Automate the transformation from raw transcript → study-ready notes with minimal human effort.

## 4. Why This Project Matters

- **Prompt engineering is a core GenAI skill** — this project teaches it through a practical lens
- **Summarization is the #1 LLM use case** in enterprise (meeting notes, report summaries, email digests)
- **Output structuring** (getting the LLM to produce JSON, markdown tables, numbered lists) is essential for building applications on top of LLMs
- **Chunking strategies** for long documents apply to any text-processing pipeline

## 5. Pipeline Architecture

```
YouTube Video URL
       │
       ▼
  [Fetch Transcript] ── youtube-transcript-api
       │
       ▼
  [Preprocess] ── merge segments, remove filler, clean whitespace
       │
       ▼
  [Chunk if needed] ── split long transcripts for context window
       │
       ▼
  [Prompt Engine] ── task-specific prompt templates
       │
       ├──▶ Summary (concise overview)
       ├──▶ Structured Notes (organized by topic)
       ├──▶ Key Takeaways (bullet points)
       ├──▶ Action Items (what to do next)
       └──▶ Quiz Questions (test understanding)
       │
       ▼
  [Combined Output] ── all formats merged into study guide
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q youtube-transcript-api langchain langchain-ollama langchain-core

print("Dependencies: youtube-transcript-api, langchain, langchain-ollama")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import warnings
import textwrap
import time
from urllib.parse import urlparse, parse_qs

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

print("All imports OK")

## 8. Configuration

In [ ]:
# ── LLM ────────────────────────────────────────────────
LLM_MODEL = "qwen3.5:9b"
LLM_TEMPERATURE = 0.3       # Slightly creative for note generation

# ── Transcript ────────────────────────────────────────
# Replace with any YouTube video URL or ID
# This is a public talk; change it to any video with subtitles
VIDEO_URL = "https://www.youtube.com/watch?v=aircAruvnKk"  # 3Blue1Brown: Neural Networks

# ── Chunking (for long transcripts) ───────────────────
MAX_CHUNK_CHARS = 6000      # Max characters per chunk sent to LLM
CHUNK_OVERLAP_CHARS = 300   # Overlap between chunks

print("Configuration loaded")
print(f"  LLM: {LLM_MODEL} (temp={LLM_TEMPERATURE})")
print(f"  Video: {VIDEO_URL}")
print(f"  Max chunk: {MAX_CHUNK_CHARS} chars")

## 9. Transcript Ingestion

The `youtube-transcript-api` fetches subtitles (auto-generated or manual) without needing a YouTube Data API key. Each segment is returned as a dict with `text`, `start` (seconds), and `duration`.

**Note:** Not all videos have transcripts. The API tries manual subtitles first, then auto-generated ones.

In [ ]:
def extract_video_id(url_or_id: str) -> str:
    """Extract YouTube video ID from a URL or return the ID itself."""
    if len(url_or_id) == 11 and not url_or_id.startswith("http"):
        return url_or_id
    parsed = urlparse(url_or_id)
    if parsed.hostname in ("youtu.be",):
        return parsed.path.lstrip("/")
    if parsed.hostname in ("www.youtube.com", "youtube.com"):
        qs = parse_qs(parsed.query)
        return qs.get("v", [url_or_id])[0]
    return url_or_id


def fetch_transcript(url_or_id: str) -> tuple[list[dict], str]:
    """Fetch transcript segments from a YouTube video.
    Returns (segments_list, video_id).
    """
    video_id = extract_video_id(url_or_id)
    print(f"Fetching transcript for video: {video_id}")

    try:
        segments = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
        print(f"  Fetched {len(segments)} segments (English)")
    except Exception:
        # Fallback: try any available language
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        transcript = next(iter(transcript_list))
        segments = transcript.fetch()
        print(f"  Fetched {len(segments)} segments (language: {transcript.language})")

    return segments, video_id


# ── Fetch the transcript ──────────────────────────────
raw_segments, video_id = fetch_transcript(VIDEO_URL)

## 10. Inspect Raw Transcript

Before processing, always look at what you're working with. Raw transcript segments are short phrases with timestamps — not clean prose.

In [ ]:
# ── Show raw segment structure ────────────────────────
print("Raw segment format:")
print(json.dumps(raw_segments[0], indent=2))

print(f"\nFirst 10 segments:")
for seg in raw_segments[:10]:
    ts = time.strftime("%M:%S", time.gmtime(seg["start"]))
    print(f"  [{ts}] {seg['text']}")

# ── Basic stats ───────────────────────────────────────
total_text = " ".join(s["text"] for s in raw_segments)
duration_sec = raw_segments[-1]["start"] + raw_segments[-1].get("duration", 0)
duration_min = duration_sec / 60

print(f"\nTranscript stats:")
print(f"  Segments: {len(raw_segments)}")
print(f"  Total characters: {len(total_text):,}")
print(f"  Total words: {len(total_text.split()):,}")
print(f"  Video duration: {duration_min:.1f} minutes")
print(f"  Avg words/minute: {len(total_text.split()) / max(duration_min, 1):.0f}")

## 11. Preprocessing

Raw YouTube transcripts have several problems:
1. **Fragmented segments** — each is a short phrase, not a sentence
2. **No punctuation** in auto-generated transcripts
3. **Filler words** — "um", "uh", "you know", "like"
4. **[Music]** and **[Applause]** annotations
5. **Repeated whitespace** and weird line breaks

We merge segments into continuous text and clean up noise. The LLM will handle capitalisation and remaining formatting.

In [ ]:
def preprocess_transcript(segments: list[dict]) -> str:
    """Clean and merge transcript segments into continuous text."""
    # Step 1: Merge all segment text
    raw = " ".join(seg["text"] for seg in segments)

    # Step 2: Remove non-speech annotations
    raw = re.sub(r"\[.*?\]", "", raw)  # [Music], [Applause], etc.

    # Step 3: Remove common filler words (case-insensitive)
    fillers = [r"\bum\b", r"\buh\b", r"\byou know\b", r"\blike\b(?=\s)",
               r"\bbasically\b", r"\bactually\b", r"\bkind of\b", r"\bsort of\b"]
    for filler in fillers:
        raw = re.sub(filler, "", raw, flags=re.IGNORECASE)

    # Step 4: Normalise whitespace
    raw = re.sub(r"\s+", " ", raw).strip()

    # Step 5: Try to restore sentence boundaries (after periods, question marks)
    raw = re.sub(r"([.!?])\s+([a-z])", lambda m: m.group(1) + " " + m.group(2).upper(), raw)

    return raw


def create_timestamped_sections(segments: list[dict], interval_sec: int = 120) -> list[dict]:
    """Group segments into time-based sections for reference."""
    sections = []
    current_section = {"start": 0, "text": ""}

    for seg in segments:
        if seg["start"] - current_section["start"] >= interval_sec and current_section["text"]:
            sections.append(current_section)
            current_section = {"start": seg["start"], "text": ""}
        current_section["text"] += " " + seg["text"]

    if current_section["text"].strip():
        sections.append(current_section)

    return sections


# ── Apply preprocessing ───────────────────────────────
transcript = preprocess_transcript(raw_segments)
sections = create_timestamped_sections(raw_segments)

print(f"Cleaned transcript: {len(transcript):,} chars, {len(transcript.split()):,} words")
print(f"Timestamped sections: {len(sections)} (2-min intervals)")
print(f"\nFirst 500 chars of cleaned transcript:")
print(transcript[:500])

## 12. Handle Long Transcripts — Chunking Strategy

LLMs have a finite context window. A 30-minute video produces ~5,000 words (~20,000 characters). If the transcript exceeds our `MAX_CHUNK_CHARS`, we split it and process each chunk separately, then merge the results.

### Two strategies for long-text summarization:

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **Map-Reduce** | Summarize each chunk independently, then summarize the summaries | Parallelizable, simple | May lose cross-chunk context |
| **Refine** | Summarize chunk 1, then refine with chunk 2, etc. | Preserves context flow | Sequential (slow), later chunks may dominate |

We use **Map-Reduce** since it's simpler and handles most transcripts well.

In [ ]:
def chunk_text(text: str, max_chars: int, overlap: int) -> list[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        if end >= len(text):
            chunks.append(text[start:])
            break

        # Find the last sentence boundary within the chunk
        for sep in [". ", "? ", "! ", "\n"]:
            last = text.rfind(sep, start, end)
            if last > start:
                end = last + len(sep)
                break

        chunks.append(text[start:end])
        start = end - overlap  # Overlap by backing up

    return chunks


# ── Chunk the transcript ──────────────────────────────
chunks = chunk_text(transcript, MAX_CHUNK_CHARS, CHUNK_OVERLAP_CHARS)

print(f"Transcript length: {len(transcript):,} chars")
print(f"Split into {len(chunks)} chunk(s)")
for i, c in enumerate(chunks):
    print(f"  Chunk {i+1}: {len(c):,} chars, {len(c.split()):,} words")

## 13. Initialize the LLM

In [ ]:
llm = ChatOllama(model=LLM_MODEL, temperature=LLM_TEMPERATURE)

# Quick test
test = llm.invoke([HumanMessage(content="Say 'ready' if you can process text.")])
response_text = test.content
if "<think>" in response_text:
    response_text = response_text.split("</think>")[-1].strip()
print(f"LLM ready: {response_text[:100]}")

## 14. Helper — Clean LLM Output

The `qwen3.5` model sometimes wraps its reasoning in `<think>` tags. We strip those for clean output.

In [ ]:
def clean_llm_output(text: str) -> str:
    """Remove thinking tags from model output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def query_llm(system: str, user: str) -> str:
    """Send a system+user message pair to the LLM and return cleaned text."""
    resp = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=user)
    ])
    return clean_llm_output(resp.content)

print("Helper functions defined")

## 15. Prompt Design — The Core Skill

This section is **the heart of the notebook**. Each note format requires a carefully designed prompt. We'll discuss **why** each prompt is structured the way it is.

### Prompt Engineering Principles

| Principle | What It Means | Example |
|-----------|--------------|----------|
| **Role setting** | Tell the LLM *who* it is | "You are an expert note-taker..." |
| **Task specification** | State exactly what output you want | "Generate 5 key takeaways..." |
| **Output format** | Describe the structure of the response | "Return as a numbered list..." |
| **Constraints** | Set boundaries on what the LLM should/shouldn't do | "Only use information from the transcript" |
| **Examples** (few-shot) | Show input/output pairs | "For example: Takeaway: ..." |
| **Grounding** | Anchor the LLM to the source material | "Based on the following transcript..." |

### 15a. Prompt 1 — Concise Summary

**Design rationale:**
- **Role:** "Expert note-taker" primes the LLM for concise, informative output
- **Length constraint:** "150-250 words" prevents both too-short and essay-length output
- **Structure hint:** Requesting an opening sentence + main topics + conclusion gives consistent format
- **Grounding:** "based ONLY on" prevents hallucinated content

In [ ]:
SUMMARY_SYSTEM = """You are an expert note-taker who creates clear, concise summaries.
Rules:
- Use ONLY information from the provided transcript
- Write in clear, professional English
- Keep the summary between 150-250 words
- Do NOT add information not present in the transcript
- Do NOT use phrases like "the speaker says" or "the video discusses"
- Write in third person present tense"""

SUMMARY_USER = """Summarize the following video transcript.

Structure your summary as:
1. Opening sentence stating the main topic
2. Key points covered (2-4 sentences)
3. Closing sentence on the overall message or conclusion

Transcript:
---
{transcript}
---

Summary:"""

print("Summary prompt template defined")
print(f"System prompt: {len(SUMMARY_SYSTEM)} chars")
print(f"User template:  {len(SUMMARY_USER)} chars (+ transcript)")

In [ ]:
# ── Generate summary ──────────────────────────────────
# For long transcripts, summarize each chunk then merge
if len(chunks) == 1:
    summary = query_llm(SUMMARY_SYSTEM, SUMMARY_USER.format(transcript=chunks[0]))
else:
    print(f"Long transcript: summarizing {len(chunks)} chunks separately, then merging...")
    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"  Summarizing chunk {i+1}/{len(chunks)}...")
        s = query_llm(SUMMARY_SYSTEM, SUMMARY_USER.format(transcript=chunk))
        chunk_summaries.append(s)

    # Merge summaries
    merged_text = "\n\n".join(chunk_summaries)
    summary = query_llm(
        SUMMARY_SYSTEM,
        f"Merge these section summaries into ONE coherent summary (150-250 words):\n\n{merged_text}\n\nFinal Summary:"
    )

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(summary)

### 15b. Prompt 2 — Structured Notes

**Design rationale:**
- **Markdown format** — produces clean, readable output that renders well in notebooks
- **"Main topics as headings"** — forces the LLM to identify and organize themes
- **Bullet points under each topic** — prevents prose-heavy output
- **"Include relevant details"** — tells the LLM not to be too terse

In [ ]:
NOTES_SYSTEM = """You are an expert at taking organized study notes from lectures.
Rules:
- Use ONLY information from the provided transcript
- Organize by topics/themes discovered in the transcript
- Use markdown formatting: ## for topics, - for bullet points
- Include specific details, examples, numbers, and definitions mentioned
- Keep each bullet point concise (1-2 sentences)
- Do NOT add external knowledge"""

NOTES_USER = """Create structured study notes from this transcript.

Format:
## Topic Name
- Key point with relevant detail
- Another point

## Next Topic
- Key point
...

Transcript:
---
{transcript}
---

Structured Notes:"""

print("Structured notes prompt defined")

In [ ]:
# ── Generate structured notes ─────────────────────────
if len(chunks) == 1:
    notes = query_llm(NOTES_SYSTEM, NOTES_USER.format(transcript=chunks[0]))
else:
    chunk_notes = []
    for i, chunk in enumerate(chunks):
        print(f"  Notes for chunk {i+1}/{len(chunks)}...")
        n = query_llm(NOTES_SYSTEM, NOTES_USER.format(transcript=chunk))
        chunk_notes.append(n)

    merged = "\n\n".join(chunk_notes)
    notes = query_llm(
        NOTES_SYSTEM,
        f"Merge and deduplicate these notes into one organized set of study notes. "
        f"Combine overlapping topics. Keep the ## heading / - bullet format.\n\n{merged}\n\nMerged Notes:"
    )

print("\n" + "=" * 60)
print("STRUCTURED NOTES")
print("=" * 60)
print(notes)

### 15c. Prompt 3 — Key Takeaways

**Design rationale:**
- **Numbered list** — easier to scan than prose, each item is self-contained
- **"5-8 takeaways"** — gives a range so the LLM can calibrate to content density
- **"Start each with a bolded phrase"** — creates a scannable format (bold = topic, rest = explanation)
- **Constraint: actionable** — pushes output from passive observation toward practical insight

In [ ]:
TAKEAWAYS_SYSTEM = """You are an expert at distilling the most important insights from content.
Rules:
- Extract only the most important, non-obvious insights
- Each takeaway should be self-contained and actionable
- Use ONLY information from the transcript
- Write clearly for someone who hasn't watched the video"""

TAKEAWAYS_USER = """Extract 5-8 key takeaways from this transcript.

Format each as:
1. **Bold Key Phrase**: Explanation in 1-2 sentences.
2. **Bold Key Phrase**: Explanation.
...

Transcript:
---
{transcript}
---

Key Takeaways:"""

# ── Generate takeaways ────────────────────────────────
# Use the full transcript (or first chunk for very long transcripts)
input_text = transcript if len(transcript) <= MAX_CHUNK_CHARS else chunks[0]
takeaways = query_llm(TAKEAWAYS_SYSTEM, TAKEAWAYS_USER.format(transcript=input_text))

print("KEY TAKEAWAYS")
print("=" * 60)
print(takeaways)

### 15d. Prompt 4 — Action Items

**Design rationale:**
- **"What should the viewer DO after watching?"** — shifts from summary to application
- **Checkbox format** — mirrors real task lists (Notion, Todoist)
- **Priority levels** — adds structure beyond a flat list
- **"Based on what was taught"** — grounds actions in the actual content

In [ ]:
ACTION_SYSTEM = """You are a learning coach who helps students create actionable study plans.
Rules:
- Derive action items ONLY from the transcript content
- Focus on what the learner should do next to apply or deepen understanding
- Be specific (not vague advice like "learn more")
- Categorize by priority"""

ACTION_USER = """Based on this transcript, generate a list of action items for the viewer.

Format:
**High Priority (do immediately):**
- [ ] Specific action item
- [ ] Another action

**Medium Priority (do this week):**
- [ ] Action item

**Low Priority (explore later):**
- [ ] Action item

Transcript:
---
{transcript}
---

Action Items:"""

# ── Generate action items ─────────────────────────────
actions = query_llm(ACTION_SYSTEM, ACTION_USER.format(transcript=input_text))

print("ACTION ITEMS")
print("=" * 60)
print(actions)

### 15e. Prompt 5 — Quiz Questions

**Design rationale:**
- **Multiple choice** — objectively gradable, good for self-testing
- **Answer + explanation** — learning happens when you understand *why* an answer is correct
- **Difficulty mix** — avoids trivial-only or impossible-only questions
- **Distractor design** — plausible wrong answers test genuine understanding, not trick knowledge

In [ ]:
QUIZ_SYSTEM = """You are an expert educator who creates fair, educational quiz questions.
Rules:
- Questions must be answerable from the transcript content ONLY
- Include a mix of recall and comprehension questions
- Wrong answers (distractors) should be plausible but clearly incorrect
- Explanations should reference what was said in the transcript"""

QUIZ_USER = """Create 5 multiple-choice quiz questions to test understanding of this transcript.

Format each question as:

**Q1: Question text here?**
A) Option A
B) Option B
C) Option C
D) Option D

**Answer: X)**
**Explanation:** Why this answer is correct based on the transcript.

---

(Repeat for Q2 through Q5)

Transcript:
---
{transcript}
---

Quiz:"""

# ── Generate quiz ─────────────────────────────────────
quiz = query_llm(QUIZ_SYSTEM, QUIZ_USER.format(transcript=input_text))

print("QUIZ QUESTIONS")
print("=" * 60)
print(quiz)

## 16. Prompt Comparison — Zero-Shot vs Few-Shot

An important prompt engineering concept: **few-shot prompting** includes examples of the desired input/output pattern. Let's compare it with the zero-shot approach we've been using.

### Why does few-shot sometimes work better?
- The LLM learns the **exact format** from examples
- Ambiguity in instructions is resolved by the example pattern
- Especially useful for structured/tabular output

### When is zero-shot sufficient?
- When the output format is simple (plain text, bullet points)
- When the instruction is unambiguous
- When context window space is limited

In [ ]:
# ── Zero-shot takeaway extraction ─────────────────────
ZERO_SHOT_PROMPT = """Extract 3 key takeaways from this transcript as a numbered list.

Transcript:
---
{transcript}
---

Takeaways:"""

# ── Few-shot takeaway extraction ──────────────────────
FEW_SHOT_PROMPT = """Extract 3 key takeaways from a transcript. Here are examples of good takeaways:

Example transcript: "Machine learning is about finding patterns in data. Supervised learning uses labeled examples. The key challenge is generalization."
Example takeaways:
1. **Pattern Recognition is Central**: ML fundamentally works by discovering patterns in data rather than following explicit rules.
2. **Labels Drive Supervised Learning**: Having labeled examples allows the algorithm to learn the mapping from inputs to outputs.
3. **Generalization > Memorization**: The true test of a model is performance on unseen data, not training accuracy.

Now extract 3 key takeaways from this transcript:
---
{transcript}
---

Takeaways:"""

# ── Compare ───────────────────────────────────────────
short_transcript = input_text[:3000]  # Use a shorter version for comparison

print("ZERO-SHOT RESULT:")
print("-" * 40)
zero_result = query_llm(
    "Extract key takeaways concisely.",
    ZERO_SHOT_PROMPT.format(transcript=short_transcript)
)
print(zero_result)

print("\nFEW-SHOT RESULT:")
print("-" * 40)
few_result = query_llm(
    "Extract key takeaways concisely.",
    FEW_SHOT_PROMPT.format(transcript=short_transcript)
)
print(few_result)

print("\nObserve: Few-shot output typically follows the bold/colon pattern")
print("more consistently because the LLM has a concrete example to mimic.")

## 17. Combined Study Guide — All Formats Together

Now we combine all generated outputs into a single, comprehensive study guide.

In [ ]:
study_guide = f"""{'=' * 70}
STUDY GUIDE
Video: https://www.youtube.com/watch?v={video_id}
Generated: {time.strftime('%Y-%m-%d %H:%M')}
{'=' * 70}

━━━ SUMMARY ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{summary}

━━━ STRUCTURED NOTES ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{notes}

━━━ KEY TAKEAWAYS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{takeaways}

━━━ ACTION ITEMS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{actions}

━━━ QUIZ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{quiz}

{'=' * 70}
END OF STUDY GUIDE
{'=' * 70}
"""

print(study_guide)

In [ ]:
# ── Save study guide to file ──────────────────────────
output_path = f"study_guide_{video_id}.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(study_guide)

print(f"Study guide saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

## 18. Reusable Pipeline Function

Let's wrap everything into a single function so you can process any YouTube video in one call.

In [ ]:
def youtube_to_notes(url_or_id: str, save: bool = True) -> dict:
    """
    Full pipeline: YouTube URL → structured study notes.

    Returns dict with: summary, notes, takeaways, actions, quiz, metadata.
    """
    t0 = time.perf_counter()

    # 1. Fetch transcript
    segments, vid = fetch_transcript(url_or_id)
    transcript_text = preprocess_transcript(segments)
    text_chunks = chunk_text(transcript_text, MAX_CHUNK_CHARS, CHUNK_OVERLAP_CHARS)
    input_text = transcript_text if len(transcript_text) <= MAX_CHUNK_CHARS else text_chunks[0]

    print(f"  Transcript: {len(transcript_text):,} chars, {len(text_chunks)} chunk(s)")

    # 2. Generate all formats
    print("  Generating summary...")
    if len(text_chunks) == 1:
        s = query_llm(SUMMARY_SYSTEM, SUMMARY_USER.format(transcript=text_chunks[0]))
    else:
        chunk_sums = [query_llm(SUMMARY_SYSTEM, SUMMARY_USER.format(transcript=c)) for c in text_chunks]
        s = query_llm(SUMMARY_SYSTEM, f"Merge into ONE summary (150-250 words):\n\n{'\n\n'.join(chunk_sums)}\n\nFinal Summary:")

    print("  Generating notes...")
    if len(text_chunks) == 1:
        n = query_llm(NOTES_SYSTEM, NOTES_USER.format(transcript=text_chunks[0]))
    else:
        chunk_ns = [query_llm(NOTES_SYSTEM, NOTES_USER.format(transcript=c)) for c in text_chunks]
        n = query_llm(NOTES_SYSTEM, f"Merge and deduplicate:\n\n{'\n\n'.join(chunk_ns)}\n\nMerged Notes:")

    print("  Generating takeaways...")
    t = query_llm(TAKEAWAYS_SYSTEM, TAKEAWAYS_USER.format(transcript=input_text))

    print("  Generating action items...")
    a = query_llm(ACTION_SYSTEM, ACTION_USER.format(transcript=input_text))

    print("  Generating quiz...")
    q = query_llm(QUIZ_SYSTEM, QUIZ_USER.format(transcript=input_text))

    elapsed = time.perf_counter() - t0

    result = {
        "video_id": vid,
        "summary": s,
        "notes": n,
        "takeaways": t,
        "actions": a,
        "quiz": q,
        "metadata": {
            "transcript_chars": len(transcript_text),
            "transcript_words": len(transcript_text.split()),
            "chunks": len(text_chunks),
            "processing_time_sec": round(elapsed, 1),
        }
    }

    if save:
        from pathlib import Path
        path = f"study_guide_{vid}.md"
        guide = f"# Study Guide — {vid}\n\n## Summary\n{s}\n\n## Notes\n{n}\n\n## Key Takeaways\n{t}\n\n## Action Items\n{a}\n\n## Quiz\n{q}\n"
        Path(path).write_text(guide, encoding="utf-8")
        print(f"  Saved to: {path}")

    print(f"  Done in {elapsed:.0f}s")
    return result

print("youtube_to_notes() pipeline function defined")

## 19. Test with a Second Video

Let's test the pipeline on a different video to verify it generalizes. Pick any video with English subtitles.

In [ ]:
# ── Try with a different video ────────────────────────
# Replace with any YouTube video ID or URL (must have subtitles)
SECOND_VIDEO = "https://www.youtube.com/watch?v=zjkBMFhNj_g"  # Andrej Karpathy: Intro to LLMs

try:
    result2 = youtube_to_notes(SECOND_VIDEO, save=False)
    print("\n" + "=" * 60)
    print("SUMMARY:")
    print(result2["summary"][:500])
    print(f"\nMetadata: {result2['metadata']}")
except Exception as e:
    print(f"Could not process second video: {e}")
    print("This is expected if the video has no available transcript.")
    print("Try replacing SECOND_VIDEO with a video you know has subtitles.")

## 20. Error Analysis & Limitations

### When this pipeline struggles

| Issue | Cause | Mitigation |
|-------|-------|------------|
| **No transcript available** | Video has subtitles disabled, or only in another language | Try `languages=["en", "en-US"]` or translate |
| **Auto-generated transcript has errors** | Speech recognition isn't perfect | LLM somewhat compensates; manual correction helps |
| **Very long videos (1h+)** | Exceeds context window even with chunking | Use map-reduce with more chunks; accept some loss |
| **Highly technical jargon** | Transcript may garble domain terms | Provide a glossary in the system prompt |
| **Multiple speakers** | No speaker attribution in auto-subs | Can't reliably attribute quotes to speakers |
| **Non-educational content** | Action items / takeaways don't apply to entertainment | Pipeline works best for lectures, tutorials, talks |
| **LLM hallucination** | Model adds plausible but unmentioned details | Strong grounding prompts reduce this; temperature=0 helps |

## 21. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **Dumping the entire transcript into one prompt** | Exceeds context window for long videos | Chunk the transcript and use map-reduce |
| **Not specifying output format in the prompt** | LLM produces inconsistent formatting every time | Always describe the desired structure explicitly |
| **Using high temperature for factual extraction** | More randomness = more hallucination | Use temp ≤ 0.3 for extraction; higher for creative tasks |
| **Skipping preprocessing** | [Music] tags, filler words, and noise waste tokens and confuse the LLM | Always clean the transcript before prompting |
| **One big prompt for everything** | Asking for "summary AND notes AND quiz" in one prompt degrades all outputs | Use separate specialized prompts for each format |
| **Not testing retrieval separately** | Can't tell if bad output is from bad transcript or bad prompting | Always inspect the raw transcript first |
| **Ignoring thinking tags** | Some models wrap internal reasoning in tags that pollute the output | Strip `<think>` tags from responses |
| **Assuming all videos have transcripts** | Many don't, or only have non-English subtitles | Always handle the exception gracefully |

## 22. How to Improve This Project

| Improvement | Difficulty | Expected Impact |
|-------------|------------|----------------|
| **Speaker diarization** (identify who said what) | Hard | Better attribution in multi-speaker content |
| **Translation support** (non-English transcripts) | Easy | Process videos in any language |
| **Timestamp-aligned notes** (link notes to video timestamps) | Medium | Jump to relevant video section from notes |
| **Custom glossary injection** (domain terms in system prompt) | Easy | Better handling of jargon |
| **Streaming output** (token-by-token display) | Easy | Better UX for long generation |
| **Batch processing** (playlist of videos) | Medium | Study guides for entire courses |
| **Flashcard export** (Anki-compatible format) | Easy | Direct import into spaced repetition apps |
| **Evaluation framework** (ROUGE, human ratings) | Medium | Systematic quality measurement |

## 23. Production Improvement Ideas

1. **Web interface** — Build a Streamlit or Gradio app: paste URL → get study guide
2. **Caching** — Cache transcripts and generated notes in a database to avoid re-processing
3. **Quality scoring** — Use a second LLM call to rate the generated notes (completeness, accuracy)
4. **User preferences** — Let users choose output formats (Cornell notes, mind maps, outlines)
5. **API endpoint** — Wrap in FastAPI: `POST /notes {url: "..."}` → returns JSON with all formats
6. **Webhook integration** — Auto-generate notes when new videos are added to a playlist
7. **PDF export** — Generate clean PDFs from the study guide markdown
8. **LLM comparison** — Test different models (GPT-4o, Claude, LLaMA) on the same transcripts

## 24. Exercises — Try These Yourself

### Exercise 1: Different Output Schema

Modify the takeaways prompt to output **JSON** instead of markdown. This is a key skill for building LLM-powered applications where you need machine-readable output.

In [ ]:
# ── Exercise 1: JSON output format ────────────────────
JSON_PROMPT = """Extract 3 key takeaways from this transcript.

Return ONLY valid JSON in this exact format:
{{
  "takeaways": [
    {{"title": "short title", "detail": "1-2 sentence explanation", "importance": "high/medium/low"}},
    ...
  ]
}}

Transcript:
---
{transcript}
---

JSON:"""

json_result = query_llm(
    "Return only valid JSON. No markdown, no explanation.",
    JSON_PROMPT.format(transcript=input_text[:3000])
)

# Try to parse the JSON
try:
    # Handle possible markdown code block
    clean = json_result.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    parsed = json.loads(clean)
    print("Parsed JSON successfully!")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"JSON parsing failed: {e}")
    print(f"Raw output: {json_result[:500]}")
    print("\nTip: Try adding 'Do NOT wrap in markdown code blocks' to the prompt")

### Exercise 2: Custom Format — Cornell Notes

Cornell Notes divide the page into three sections: **Cue Column** (questions/keywords), **Note-Taking Area** (details), **Summary** (bottom). Try creating a prompt that produces this format.

In [ ]:
# ── Exercise 2: Cornell Notes format ──────────────────
CORNELL_PROMPT = """Create Cornell-style notes from this transcript.

Format:
| Cue (Question/Keyword) | Notes (Details) |
|---|---|
| Keyword or question | Detailed notes from the transcript |
| ... | ... |

**Summary (2-3 sentences):**
Overall summary of the content.

Transcript:
---
{transcript}
---

Cornell Notes:"""

cornell = query_llm(
    "Create structured Cornell notes. Use a markdown table.",
    CORNELL_PROMPT.format(transcript=input_text[:3000])
)

print("CORNELL NOTES OUTPUT:")
print("=" * 60)
print(cornell)

### Mini Challenge

1. **Process your own video:** Replace `VIDEO_URL` with a lecture or talk you've recently watched. Does the study guide help you review?

2. **Add a "Key Vocabulary" prompt:** Design a prompt that extracts technical terms and their definitions from the transcript. Output as a glossary table.

3. **Implement the Refine strategy:** Instead of map-reduce, implement the refine approach: summarize chunk 1, then "refine this summary with additional context from chunk 2", etc. Compare results.

4. **Add timestamp links:** Modify the notes output to include `[MM:SS]` timestamps from the original segments, so readers can jump to the relevant part of the video.

5. **Evaluate quality:** Run the pipeline on 3 different videos and rate each output (1-10) on completeness, accuracy, and usefulness. What patterns do you see?

## 25. Session Statistics

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)
print(f"Video processed: https://www.youtube.com/watch?v={video_id}")
print(f"Transcript: {len(transcript):,} chars, {len(transcript.split()):,} words")
print(f"Chunks: {len(chunks)}")
print(f"LLM: {LLM_MODEL}")
print(f"\nGenerated outputs:")
print(f"  Summary:    {len(summary):,} chars")
print(f"  Notes:      {len(notes):,} chars")
print(f"  Takeaways:  {len(takeaways):,} chars")
print(f"  Actions:    {len(actions):,} chars")
print(f"  Quiz:       {len(quiz):,} chars")
total_output = len(summary) + len(notes) + len(takeaways) + len(actions) + len(quiz)
print(f"  Total:      {total_output:,} chars")
compression = len(transcript) / max(total_output, 1)
print(f"\nCompression ratio: {compression:.1f}x (transcript → outputs)")

## 26. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Prompt design is the primary skill** — the same LLM produces dramatically different output with different prompts |
| 2 | **Separate prompts for separate tasks** — one prompt per output format works better than one mega-prompt |
| 3 | **Always specify output format** — tell the LLM exactly what structure you want (bullets, tables, JSON) |
| 4 | **Preprocessing matters** — cleaning transcript noise improves LLM output and saves tokens |
| 5 | **Chunking enables scale** — map-reduce lets you process transcripts of any length |
| 6 | **Few-shot > zero-shot for formatting** — examples in the prompt make output more consistent |
| 7 | **Grounding reduces hallucination** — "ONLY use information from the transcript" is essential |
| 8 | **Test edge cases** — videos without transcripts, very long content, and non-English sources all need handling |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*